In [1]:
import cv2
import numpy as np
import os
from PIL import Image

In [2]:
def createUser(fid, name):
    web = cv2.VideoCapture(0)
    web.set(3,640)
    web.set(4,480)

    faces = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    f_dir = 'dataset'
    f_name = name
    path = os.path.join(f_dir, f_name)
    if not os.path.isdir(path):
        os.makedirs(path, exist_ok=True)

    counter = 0
    while(True):
        ret, img = web.read()
        img = cv2.flip(img,1)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        multi_face = faces.detectMultiScale(gray, 1.3, 5)
        for (x,y,w,h) in multi_face:
            cv2.rectangle(img, (x,y), (x+w,y+h), (0,0,0), thickness = 3)
            counter+=1
            
            cv2.imwrite("{}/{}.{}.{}{}".format(path, name, fid, counter, ".jpg"), gray[y:y+h, x:x+w])
            cv2.imshow("IMAGES", img)
        k = cv2.waitKey(100) & 0xff
        if k ==27:
            break
        elif counter>=25:
            break
    web.release()
    cv2.destroyAllWindows()

In [3]:
createUser(1, "Aanya")

In [4]:
def train():
    database = 'dataset'
    img_dir = [x[0] for x in os.walk(database)][1::]
    recognizer = cv2.face.LBPHFaceRecognizer_create() #Local Binary Pikture histogram
    detector = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faceSamples = []
    ids = []
    for path in img_dir:
        path = str(path)
        imagePaths = [os.path.join(path, f) for f in os.listdir(path)]

        for imagePath in imagePaths:
            PIL_img = Image.open(imagePath).convert('L')
            img_numpy = np.array(PIL_img, 'uint8')
            id = int(os.path.split(imagePath)[-1].split('.')[1])
            faces = detector.detectMultiScale(img_numpy)

            for (x,y,w,h) in faces:
                faceSamples.append(img_numpy[y:y+h , x:x+w])
                ids.append(id)

    recognizer.train(faceSamples, np.array(ids))
    recognizer.write('trainer.yml')
    print('\n [INFO] {0} faces trained. Existing Program'.format(len(np.unique(ids))))
    return len(np.unique(ids))

In [5]:
train()


 [INFO] 1 faces trained. Existing Program


1

In [6]:
def recongize(names):
    recognizer = cv2.face.LBPHFaceRecognizer_create()
    recognizer.read('trainer.yml')
    faceCascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

    font = cv2.FONT_HERSHEY_SIMPLEX
    id = 0
    name = ""
    face_count = 0

    cap = cv2.VideoCapture(0)
    cap.set(3,640)
    cap.set(4,480)

    minW = 0.1%cap.get(3)
    minH = 0.1%cap.get(4)
    while True:
        ret, img = cap.read()
        img = cv2.flip(img,1)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        faces = faceCascade.detectMultiScale(gray, 1.1, 5, minSize = (int(minW), int(minH)))

        for (x,y,w,h) in faces:
            cv2.rectangle(img, (x,y), (x+w , y+h), (255,255,255), 2)

            id, confidence = recognizer.predict(gray[y:y+h, x:x+w])
            if(confidence<70):
                id = names[id]
            else:
                id = 'unknown'
                roll = None
                confidence = "{0}%".format(round(100-confidence))
                text = "Not recognized"

            if name == id:
                face_count+=1
                if face_count>21:
                    found_count = -100
            else:
                name = id
                face_count =0
            cv2.putText(img,str(id),(x+5, y-5), font, 1, (255,255,255), 2)
            cv2.putText(img,str(confidence),(x+5, y+h-5), font, 1, (255,255,0), 1)

        cv2.imshow("CAM", img)
        k = cv2.waitKey(5) & 0xff
        if k == 27:
            break
    cap.release()
    cap.destroyAllWindows()

In [ ]:
recongize({1: "Aanya"})